In [40]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix


In [41]:
df=pd.read_csv(r'C:\Users\fathi\OneDrive\Desktop\5thsem_OJT\project_diabetes\data\processed\risvana_p1w2_features.csv')

In [42]:
df.head()

,patient_id,age,gender,pregnancies,glucose,blood_pressure,skin_thickness,insulin,bmi,diabetes_pedigree,...,smoker,last_checkup_date,outcome,age band,bmi category,glucose category,insulin_glucose_ratio,pregnancies_age_rate,days_since_checkup,smoker_activity
0,P00826,79,male,0,124.0,71.0,30.0,124.0,23.4,0.588,...,NO,2023-04-20,0,over 55,normal,prediabetic,1.000000,0.000000,618,Unknown/NO
1,P00108,37,male,0,153.0,85.0,24.0,42.0,33.9,0.192,...,YES,2024-05-03,1,35-55,obese,diabetic,0.274510,0.000000,239,Medium/YES
2,P00517,39,female,0,142.0,68.0,22.0,159.0,26.9,0.777,...,UNKNOWN,2023-07-08,0,35-55,over weight,diabetic,1.119718,0.000000,539,Low/UNKNOWN
3,P00687,68,female,7,121.0,88.0,26.0,124.0,29.0,1.217,...,YES,2024-04-14,1,over 55,over weight,prediabetic,1.024793,0.102941,258,Unknown/YES
4,P00927,75,female,0,107.0,84.0,31.0,101.0,21.2,0.278,...,NO,2024-01-27,0,over 55,normal,prediabetic,0.943925,0.000000,336,Low/NO


In [43]:

feature_cols = [
    'age',
    'pregnancies',
    'glucose',
    'blood_pressure',
    'skin_thickness',
    'insulin',
    'bmi',
    'diabetes_pedigree',
    'bmi category',
    'glucose category'
]

In [44]:
x = df[feature_cols].copy()
y = df["outcome"]


In [45]:
                                                               
x=pd.get_dummies(x,drop_first=True)

In [46]:
x_train,x_test,y_train,y_test= train_test_split(
x,y,test_size=0.2,random_state=42,stratify=y)

In [47]:
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [48]:
logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled, y_train)
dec_tree=DecisionTreeClassifier(max_depth=3).fit(x_train,y_train)
knn_best = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)

In [49]:
print('LogisticRegression',accuracy_score(y_test,logreg.predict(x_test_scaled)))
print('DecisionTreeClassifier',accuracy_score(y_test,dec_tree.predict(x_test)))
print('kneighborsclassifier',accuracy_score(y_test,knn_best.predict(x_test_scaled)))

LogisticRegression 0.7526315789473684
DecisionTreeClassifier 0.7210526315789474
kneighborsclassifier 0.7473684210526316


In [50]:
print('test set:',len(y_test),'patients |',int(np.sum(y_test)),'of them diabetic')

test set: 190 patients | 55 of them diabetic


In [51]:
models=[('logisticregression',logreg,x_test_scaled),
        ('DecisionTreeClassifier(d=3)',dec_tree,x_test),
        ('knn(k=25)',knn_best,x_test_scaled)]

In [52]:
rows=[]
for name,m,xt in models:
    cm=confusion_matrix(y_test,m.predict(xt))
    tn,fp,fn,tp=cm.ravel()
    rows.append({"model":name,"correct_negatives":int(tn),"false_alarams":int(fp),
              "missed_patients":int(fn),"found_patients":int(tp),
              "accuracy": round(accuracy_score(y_test,m.predict(xt)),4)})
matrix=pd.DataFrame(rows)
matrix


,model,correct_negatives,false_alarams,missed_patients,found_patients,accuracy
0,logisticregression,127,8,39,16,0.7526
1,DecisionTreeClassifier(d=3),121,14,39,16,0.7211
2,knn(k=25),132,3,45,10,0.7474


                        logistic regression
* **127 (TN):** 127 people were correctly identified as not having diabetes.
* **8 (FP):** 8 people were incorrectly identified as having diabetes.
* **39 (FN):** 39 people who actually had diabetes were missed by the model.
* **16 (TP):** 16 people who had diabetes were correctly identified by the model.


* **Logistic Regression:** The main error is **missed patients (FN = 39)**.
* **Decision Tree:** The main error is **missed patients (FN = 39)**, along with 14 false alarms.
* **KNN:** The main error is **missed patients (FN = 45)**, which is the highest among the three models.



score the model that never says yes

In [53]:
from sklearn.dummy import DummyClassifier


In [54]:
dummy=DummyClassifier(strategy='most_frequent')
dummy.fit(x_train,y_train)
y_pred=dummy.predict(x_test)
confusion_matrix(y_test,y_pred)

array([[135,   0],
       [ 55,   0]])

In [55]:
print('dummyclassifier',accuracy_score(y_test,y_pred))

dummyclassifier 0.7105263157894737


In [56]:
y_test.value_counts(normalize=True)

outcome
0    0.710526
1    0.289474
Name: proportion, dtype: float64

# precision and recall


Recall → How many actual positives did I find?
Formula: TP / (TP + FN)

In [57]:
cm=confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print("recall : tp/tp+fn", tp/(tp+fn))

recall : tp/tp+fn 0.2909090909090909


In [58]:
from sklearn.metrics import recall_score
recall_score(y_test,logreg.predict(x_test_scaled))

0.2909090909090909

In [59]:
recall_score(y_test,logreg.predict(x_test_scaled))
print("DecisionTreeClassifier  Recall =",recall_score(y_test,dec_tree.predict(x_test)))
print("KNeighborsClassifier  Recall =",recall_score(y_test,knn_best.predict(x_test_scaled)))
print("LogisticRegression  recall=",recall_score(y_test,logreg.predict(x_test_scaled)))

DecisionTreeClassifier  Recall = 0.2909090909090909
KNeighborsClassifier  Recall = 0.18181818181818182
LogisticRegression  recall= 0.2909090909090909


# precision

Precision = How correct are my positive predictions?
Formula: TP / (TP + FP)

In [60]:
confusion_matrix(y_test,logreg.predict(x_test_scaled))
tn,fp,fn,tp=cm.ravel()
print("precision : tp/tp+f", tp/(tp+fp))

precision : tp/tp+f 0.6666666666666666


In [61]:
from sklearn.metrics import precision_score
precision_score(y_test,logreg.predict(x_test_scaled))

0.6666666666666666

In [ ]:
precision_score(y_test,logreg.predict(x_test_scaled))
print("DecisionTreeClassifier  precision =",precision_score(y_test,dec_tree.predict(x_test)))
print("KNeighborsClassifier  precision =",precision_score(y_test,knn_best.predict(x_test_scaled)))
print("LogisticRegression  precision=",precision_score(y_test,logareg.predict(x_test_scaled)))

DecisionTreeClassifier  precision = 0.5333333333333333
KNeighborsClassifier  precision = 0.7692307692307693
LogisticRegression  precision= 0.6666666666666666
